# Day 18 — Short-term memory

An agent's context window is finite, so you can't keep every turn forever. The classic
fix: keep a **rolling buffer** of recent turns, and when it overflows, **summarize** the
oldest turns into a running summary you prepend to the system prompt.

You'll implement a tiny `Memory` class with `add()`, automatic summarization, and
`as_messages()` to feed the model.

In [1]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

repo root on sys.path: c:\autogen\AILearning


## 🔬 Your turn

Fill in the `TODO`s, then run the cell. Stuck? The solution is below — but try first.

In [ ]:
from shared.llm import chat

class Memory:
    def __init__(self, max_turns=6):
        self.turns = []        # recent [{"role", "content"}]
        self.summary = ""      # compressed older history
        self.max_turns = max_turns

    def add(self, role, content):
        self.turns.append({"role": role, "content": content})
        self._maybe_summarize()

    def _maybe_summarize(self):
        # TODO: if len(self.turns) > self.max_turns:
        #   overflow = len(self.turns) - self.max_turns
        #   take the oldest `overflow` turns, ask the model to fold them into self.summary
        #   (<= 3 sentences), and keep only the most recent max_turns turns.
        pass

    def as_messages(self, system):
        sys = system + (f"\n\nConversation summary so far: {self.summary}" if self.summary else "")
        return [{"role": "system", "content": sys}] + self.turns

# m = Memory(max_turns=4)
# for i in range(6):
#     m.add("user", f"fact {i}")
# print("summary:", m.summary, "| kept:", len(m.turns))

## 🔒 Solution

One correct way to do it. Compare it with your version.

In [ ]:
from shared.llm import chat

class Memory:
    def __init__(self, max_turns=6):
        self.turns = []
        self.summary = ""
        self.max_turns = max_turns

    def add(self, role, content):
        self.turns.append({"role": role, "content": content})
        self._maybe_summarize()

    def _maybe_summarize(self):
        overflow = len(self.turns) - self.max_turns
        if overflow <= 0:
            return
        old, self.turns = self.turns[:overflow], self.turns[overflow:]
        transcript = "\n".join(f"{t['role']}: {t['content']}" for t in old)
        self.summary = chat([
            {"role": "system", "content": "Fold the new lines into the running summary. Keep it under 3 sentences."},
            {"role": "user", "content": f"Running summary: {self.summary or '(none)'}\n\nNew lines:\n{transcript}"},
        ], temperature=0)

    def as_messages(self, system):
        sys = system + (f"\n\nConversation summary so far: {self.summary}" if self.summary else "")
        return [{"role": "system", "content": sys}] + self.turns

m = Memory(max_turns=4)
for i in range(6):
    m.add("user", f"fact {i}: my lucky number includes {i}")
print("summary:", m.summary)
print("kept turns:", len(m.turns))